In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))


In [4]:
print(sys.path)

['C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\python311.zip', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\DLLs', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\Lib', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting\\penv', '', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting\\penv\\Lib\\site-packages', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting']


In [5]:
# IMPORTING DEFINED FUNCTIONS

from src.text_cleaning import (
    standardise_columns,
    normalize_text_column,
    normalize_indian_state_names,
    normalize_district_name
)

from src.date_utils import (
    extract_calendar_year,
    extract_month_name,
    calculate_financial_year_from_month_raw,
    create_month_start_date,
    extract_month_number,
    extract_quarter
)

from src.geo_utils import (
    map_country_to_code,
    map_state_codes,
    expand_directional_districts,
    map_district_codes
)


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120) 

In [7]:
DATA_PATH = "../data/raw/"
ADDITIONAL_DATA_PATH = "../data/raw/additional_data/" 

In [8]:
# LOADING DATASETS

df_dist = pd.read_csv(
    DATA_PATH + "National Food Security Act Monthly Allocation Transaction and Foodgrain Distribution.csv"
)

df_port = pd.read_csv(
    DATA_PATH + "Number of Transactions Enabled through Ration Card Portability.csv"
)

df_epos = pd.read_csv(
    DATA_PATH + "Electronic Point of Sale Transactions - Aadhaar Authenticated and other Mode of Authentication.csv"
)

In [9]:
df_dist.shape

(2357, 12)

In [10]:
df_dist.sample(5)

,Country,State,Year,Month,"Food Grains Allocated (UOM:MT(MetricTonne)), Scaling Factor:1000","Ration Cards Issued (UOM:Number), Scaling Factor:1","Transaction For Ration Cards (UOM:Number), Scaling Factor:1","Aadhaar Authenticated Transactions (UOM:Number), Scaling Factor:1","Aadhaar Authenticated Transactions (%) (UOM:%(Percentage)), Scaling Factor:1","Epos (Electronic Point Of Sale System) Distribution Of Food Grains (UOM:MT(MetricTonne)), Scaling Factor:1000","Manual Distribution Of Food Grains (UOM:MT(MetricTonne)), Scaling Factor:1000","Distribution Of Food Grains (UOM:MT(MetricTonne)), Scaling Factor:1000"
1928,India,The Dadra And Nagar Haveli And Daman And Diu,"Calendar Year (Jan - Dec), 2018","October, 2018",1380.930000,5.435100e+04,5.247900e+04,4.588000e+04,91.980782,1221.820000,0.000000,1221.820000
953,India,Jammu And Kashmir,"Calendar Year (Jan - Dec), 2021","March, 2021",38927.541683,1.344836e+06,1.472399e+06,1.143643e+06,77.670000,33953.893756,10.162738,33964.046713
918,India,Haryana,"Calendar Year (Jan - Dec), 2021","April, 2021",66250.000000,2.695639e+06,2.367538e+06,2.367538e+06,100.000000,59010.500000,0.000000,59010.500000
2282,India,Odisha,"Calendar Year (Jan - Dec), 2017","November, 2017",179926.060000,8.629658e+06,1.177180e+07,3.327055e+06,28.260000,168267.510000,0.000000,168267.510000
323,India,Maharashtra,"Calendar Year (Jan - Dec), 2023","April, 2023",383766.000000,1.559218e+07,1.464924e+07,1.449926e+07,98.980000,338911.210000,0.000000,338911.210000


In [11]:
df_dist=standardise_columns(df_dist)

In [12]:
df_dist.columns

Index(['country', 'state', 'year', 'month', 'food_grains_allocated_uommtmetrictonne_scaling_factor1000',
       'ration_cards_issued_uomnumber_scaling_factor1', 'transaction_for_ration_cards_uomnumber_scaling_factor1',
       'aadhaar_authenticated_transactions_uomnumber_scaling_factor1',
       'aadhaar_authenticated_transactions_uompercentage_scaling_factor1',
       'epos_electronic_point_of_sale_system_distribution_of_food_grains_uommtmetrictonne_scaling_factor1000',
       'manual_distribution_of_food_grains_uommtmetrictonne_scaling_factor1000',
       'distribution_of_food_grains_uommtmetrictonne_scaling_factor1000'],
      dtype='object')

In [13]:
# RENAMING COLUMNS

df_dist = df_dist.rename(columns={
     "year": "year_raw",
     "month": "month_raw",
     "food_grains_allocated_uommtmetrictonne_scaling_factor1000": "foodgrains_allocated",
     "ration_cards_issued_uomnumber_scaling_factor1": "rationcards_issued",
     "transaction_for_ration_cards_uomnumber_scaling_factor1": "trans_for_rationcards",
     "aadhaar_authenticated_transactions_uomnumber_scaling_factor1": "aadhaar_auth_trans",
     "aadhaar_authenticated_transactions_uompercentage_scaling_factor1": "aadhaar_trans_percent",
     "epos_electronic_point_of_sale_system_distribution_of_food_grains_uommtmetrictonne_scaling_factor1000": "epos_distributed_qty",
     "manual_distribution_of_food_grains_uommtmetrictonne_scaling_factor1000": "manual_distributed_qty",
     "distribution_of_food_grains_uommtmetrictonne_scaling_factor1000": "distributed_qty"
 })

In [14]:
df_port.shape

(1776, 13)

In [15]:
df_port.sample(5)

,Country,State,Year,Month,"Ration Cards (UOM:Number), Scaling Factor:1","Transactions Through Portability-Inter District Under Antyodaya Anna Yojana (UOM:Number), Scaling Factor:1","Transactions Through Portability-Inter District Under Priority Household (UOM:Number), Scaling Factor:1","Transactions Through Portability-Intra District Under Antyodaya Anna Yojana (UOM:Number), Scaling Factor:1","Transactions Through Portability-Intra District Under Priority Household (UOM:Number), Scaling Factor:1","Quantity Distributed Through Portability Inter Distric Under Antyodaya Anna Yojana (UOM:MT(MetricTonne)), Scaling Factor:1","Quantity Distributed Through Portability Inter District Under Priority Household (UOM:MT(MetricTonne)), Scaling Factor:1","Quantity Distributed Through Portability Intra District Under Antyodaya Anna Yojana (UOM:MT(MetricTonne)), Scaling Factor:1","Quantity Distributed Through Portability Intra District Under Priority Household (UOM:MT(MetricTonne)), Scaling Factor:1"
1159,India,Haryana,"Calendar Year (Jan - Dec), 2019","August, 2019",2669503.0,2376.0,10222.0,55836.0,511378.0,60.79,179.53,1510.03,10505.47
598,India,Lakshadweep,"Calendar Year (Jan - Dec), 2021","March, 2021",5158.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
543,India,Mizoram,"Calendar Year (Jan - Dec), 2021","May, 2021",161651.0,1.0,2.0,7.0,34.0,0.04,0.03,0.25,0.89
852,India,Uttar Pradesh,"Calendar Year (Jan - Dec), 2020","July, 2020",35565150.0,2745.0,53008.0,29912.0,679226.0,96.07,1087.41,1046.92,14148.72
1550,India,Punjab,"Calendar Year (Jan - Dec), 2018","June, 2018",3515987.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00


In [16]:
df_port=standardise_columns(df_port)

In [17]:
df_port.columns

Index(['country', 'state', 'year', 'month', 'ration_cards_uomnumber_scaling_factor1',
       'transactions_through_portabilityinter_district_under_antyodaya_anna_yojana_uomnumber_scaling_factor1',
       'transactions_through_portabilityinter_district_under_priority_household_uomnumber_scaling_factor1',
       'transactions_through_portabilityintra_district_under_antyodaya_anna_yojana_uomnumber_scaling_factor1',
       'transactions_through_portabilityintra_district_under_priority_household_uomnumber_scaling_factor1',
       'quantity_distributed_through_portability_inter_distric_under_antyodaya_anna_yojana_uommtmetrictonne_scaling_factor1',
       'quantity_distributed_through_portability_inter_district_under_priority_household_uommtmetrictonne_scaling_factor1',
       'quantity_distributed_through_portability_intra_district_under_antyodaya_anna_yojana_uommtmetrictonne_scaling_factor1',
       'quantity_distributed_through_portability_intra_district_under_priority_household_uommtmetri

In [18]:
# RENAMING COLUMNS
# RENAMING THE PORTABILITY DATAFRAME COLUMNS FOR EASY READABILITY
df_port = df_port.rename(columns={
     "year": "year_raw",
     "month": "month_raw",
     "ration_cards_uomnumber_scaling_factor1": "rationcards_issued",
     "transactions_through_portabilityinter_district_under_antyodaya_anna_yojana_uomnumber_scaling_factor1": "interdistrict_port_aay_trans",
     "transactions_through_portabilityinter_district_under_priority_household_uomnumber_scaling_factor1": "interdistrict_port_pyh_trans",
     "transactions_through_portabilityintra_district_under_antyodaya_anna_yojana_uomnumber_scaling_factor1": "intradistrict_port_aay_trans",
     "transactions_through_portabilityintra_district_under_priority_household_uomnumber_scaling_factor1": "intradistrict_port_pyh_trans",
     "quantity_distributed_through_portability_inter_distric_under_antyodaya_anna_yojana_uommtmetrictonne_scaling_factor1": "interdistrict_port_aay_qty",
     "quantity_distributed_through_portability_inter_district_under_priority_household_uommtmetrictonne_scaling_factor1": "interdistrict_port_pyh_qty",
     "quantity_distributed_through_portability_intra_district_under_antyodaya_anna_yojana_uommtmetrictonne_scaling_factor1": "intradistrict_port_aay_qty",
     "quantity_distributed_through_portability_intra_district_under_priority_household_uommtmetrictonne_scaling_factor1": "intradistrict_port_pyh_qty"
 })

In [19]:
df_port.columns

Index(['country', 'state', 'year_raw', 'month_raw', 'rationcards_issued', 'interdistrict_port_aay_trans',
       'interdistrict_port_pyh_trans', 'intradistrict_port_aay_trans', 'intradistrict_port_pyh_trans',
       'interdistrict_port_aay_qty', 'interdistrict_port_pyh_qty', 'intradistrict_port_aay_qty',
       'intradistrict_port_pyh_qty'],
      dtype='object')

In [20]:
df_epos.shape

(2017, 18)

In [21]:
df_epos.sample(5)

,Country,State,Year,Month,"Ration Card Holders Under National Food Security Act (Nfsa) (UOM:Number), Scaling Factor:1","Ration Cards Involved In The Transactions Under National Food Security Act (Nfsa) (UOM:Number), Scaling Factor:1","Electronic Point Of Sale (Epos) Transactions Under National Food Security Act (Nfsa) (UOM:Number), Scaling Factor:1","Aadhar Based Biometric Transactions (UOM:Number), Scaling Factor:1","Aadhar Based Integrated Risk Information System (Iris) Transactions (UOM:Number), Scaling Factor:1","Aadhar Based One Time Password (Otp) Transactions (UOM:Number), Scaling Factor:1","Total Aadhaar Based Transactions (UOM:Number), Scaling Factor:1","Total Aadhaar Transactions (%) (UOM:%(Percentage)), Scaling Factor:1","Transactions Authenticated By The Public Distribution System (Pds) And One Time Password (Otp) (UOM:Number), Scaling Factor:1","Transactions Authenticated By Modes Other Than One Time Password (Otp) (UOM:Number), Scaling Factor:1","Authenticated Transactions By Other Mode (UOM:Number), Scaling Factor:1","Authenticated Transactions By Other Mode (%) (UOM:%(Percentage)), Scaling Factor:1","Non-Authenticated Transactions (UOM:Number), Scaling Factor:1","Non-Authenticated Transactions (%) (UOM:%(Percentage)), Scaling Factor:1"
547,India,Delhi,"Calendar Year (Jan - Dec), 2022","September, 2022",1.782909e+06,1.729104e+06,1.748275e+06,1.658530e+06,89745.0,0.0,1.748275e+06,100.0,0.0,0.0,0.0,0.0,0.000000,0.0
817,India,Tamil Nadu,"Calendar Year (Jan - Dec), 2021","April, 2021",1.119936e+07,1.046056e+07,1.081622e+07,6.608056e+06,0.0,0.0,6.608056e+06,61.0,0.0,4162107.0,4162107.0,38.0,46057.000000,0.0
1811,India,Ladakh,"Calendar Year (Jan - Dec), 2018","May, 2018",3.646900e+04,2.314360e+04,2.328464e+04,2.607388e+03,0.0,0.0,2.607388e+03,11.0,0.0,0.0,0.0,0.0,20677.247897,88.0
958,India,Lakshadweep,"Calendar Year (Jan - Dec), 2020","November, 2020",5.160000e+03,2.119000e+03,2.124000e+03,0.000000e+00,0.0,0.0,0.000000e+00,0.0,0.0,0.0,0.0,0.0,2124.000000,100.0
1129,India,Goa,"Calendar Year (Jan - Dec), 2020","May, 2020",1.424940e+05,1.506540e+05,1.939280e+05,1.878840e+05,723.0,0.0,1.886070e+05,97.0,0.0,0.0,0.0,0.0,5321.000000,2.0


In [22]:
df_epos=standardise_columns(df_epos)

In [23]:
df_epos.columns

Index(['country', 'state', 'year', 'month',
       'ration_card_holders_under_national_food_security_act_nfsa_uomnumber_scaling_factor1',
       'ration_cards_involved_in_the_transactions_under_national_food_security_act_nfsa_uomnumber_scaling_factor1',
       'electronic_point_of_sale_epos_transactions_under_national_food_security_act_nfsa_uomnumber_scaling_factor1',
       'aadhar_based_biometric_transactions_uomnumber_scaling_factor1',
       'aadhar_based_integrated_risk_information_system_iris_transactions_uomnumber_scaling_factor1',
       'aadhar_based_one_time_password_otp_transactions_uomnumber_scaling_factor1',
       'total_aadhaar_based_transactions_uomnumber_scaling_factor1',
       'total_aadhaar_transactions_uompercentage_scaling_factor1',
       'transactions_authenticated_by_the_public_distribution_system_pds_and_one_time_password_otp_uomnumber_scaling_factor1',
       'transactions_authenticated_by_modes_other_than_one_time_password_otp_uomnumber_scaling_factor1',
   

In [24]:
# RENAMING COLUMNS
# RENAMING THE EPOS DATAFRAME COLUMNS FOR EASY READABILITY
df_epos = df_epos.rename(columns={
     "year": "year_raw",
     "month": "month_raw",
     "ration_card_holders_under_national_food_security_act_nfsa_uomnumber_scaling_factor1": "rationcard_holders",
     "ration_cards_involved_in_the_transactions_under_national_food_security_act_nfsa_uomnumber_scaling_factor1": "rationcards_in_trans",
     "electronic_point_of_sale_epos_transactions_under_national_food_security_act_nfsa_uomnumber_scaling_factor1": "epos_trans",
     "aadhar_based_biometric_transactions_uomnumber_scaling_factor1": "biometric_trans",
     "aadhar_based_integrated_risk_information_system_iris_transactions_uomnumber_scaling_factor1": "iris_trans",
     "aadhar_based_one_time_password_otp_transactions_uomnumber_scaling_factor1": "otp_trans",
     "total_aadhaar_based_transactions_uomnumber_scaling_factor1": "total_aadhaar_trans",
     "total_aadhaar_transactions_uompercentage_scaling_factor1": "total_aadhaar_trans_percent",
     "transactions_authenticated_by_the_public_distribution_system_pds_and_one_time_password_otp_uomnumber_scaling_factor1": "otp_auth_trans",
     "transactions_authenticated_by_modes_other_than_one_time_password_otp_uomnumber_scaling_factor1": "not_otp_auth_trans",
     "authenticated_transactions_by_other_mode_uomnumber_scaling_factor1": "other_mode_auth_trans",
     "authenticated_transactions_by_other_mode_uompercentage_scaling_factor1": "other_mode_auth_trans_percent",
     "nonauthenticated_transactions_uomnumber_scaling_factor1": "non_auth_trans",
     "nonauthenticated_transactions_uompercentage_scaling_factor1": "non_auth_trans_percent"   
 })


In [25]:
df_epos.columns

Index(['country', 'state', 'year_raw', 'month_raw', 'rationcard_holders', 'rationcards_in_trans', 'epos_trans',
       'biometric_trans', 'iris_trans', 'otp_trans', 'total_aadhaar_trans', 'total_aadhaar_trans_percent',
       'otp_auth_trans', 'not_otp_auth_trans', 'other_mode_auth_trans', 'other_mode_auth_trans_percent',
       'non_auth_trans', 'non_auth_trans_percent'],
      dtype='object')

In [26]:
for df in [df_dist, df_port, df_epos]:
    print(df.duplicated().sum())

0
0
0


In [27]:
for df in [df_dist, df_port, df_epos]:
    print(df.isna().sum())
    print("-"*100)
    print("\n")

country                   0
state                     0
year_raw                  0
month_raw                 0
foodgrains_allocated      0
rationcards_issued        0
trans_for_rationcards     0
aadhaar_auth_trans        0
aadhaar_trans_percent     0
epos_distributed_qty      0
manual_distributed_qty    0
distributed_qty           0
dtype: int64
----------------------------------------------------------------------------------------------------


country                         0
state                           0
year_raw                        0
month_raw                       0
rationcards_issued              0
interdistrict_port_aay_trans    0
interdistrict_port_pyh_trans    0
intradistrict_port_aay_trans    0
intradistrict_port_pyh_trans    0
interdistrict_port_aay_qty      0
interdistrict_port_pyh_qty      0
intradistrict_port_aay_qty      0
intradistrict_port_pyh_qty      0
dtype: int64
---------------------------------------------------------------------------------------------

In [28]:
for df in [df_dist, df_port, df_epos]:
    print(df.shape)

(2357, 12)
(1776, 13)
(2017, 18)


In [29]:
for df in [df_dist, df_port, df_epos]:
    print(df.describe().T)
    print("-"*100)
    print("\n")

                         count          mean           std  min        25%         50%         75%          max
foodgrains_allocated    2357.0  1.355349e+05  1.758428e+05  0.0    7632.11    72510.00   202114.13   1623596.68
rationcards_issued      2357.0  6.812986e+06  1.080195e+07  0.0  180490.00  2961299.00  9313435.00  56360939.00
trans_for_rationcards   2357.0  4.862639e+06  6.647375e+06  0.0   47791.00  1541708.00  8619968.00  34446280.00
aadhaar_auth_trans      2357.0  3.956190e+06  6.267435e+06  0.0    5896.00   579580.00  5901449.00  34305361.00
aadhaar_trans_percent   2357.0  6.194871e+01  4.303067e+01  0.0       6.87       89.50       99.98       100.00
epos_distributed_qty    2357.0  1.009128e+05  1.507528e+05  0.0     750.19    27119.86   167652.95   1547320.53
manual_distributed_qty  2357.0  1.893522e+04  8.992039e+04  0.0       0.00        0.00     2614.17   2484367.22
distributed_qty         2357.0  1.198480e+05  1.723157e+05  0.0    4429.89    52793.65   178693.30   261

In [30]:
for df in [df_dist, df_port, df_epos]:
    print(df.describe(include='O').T)
    print("-"*100)
    print("\n")

          count unique                              top  freq
country    2357      1                            India  2357
state      2357     34                           Kerala    79
year_raw   2357      8  Calendar Year (Jan - Dec), 2018   404
month_raw  2357     81                      March, 2018    34
----------------------------------------------------------------------------------------------------


          count unique                              top  freq
country    1776      1                            India  1776
state      1776     34                   Andhra Pradesh    62
year_raw   1776      6  Calendar Year (Jan - Dec), 2021   378
month_raw  1776     62                   February, 2022    34
----------------------------------------------------------------------------------------------------


          count unique                              top  freq
country    2017      1                            India  2017
state      2017     34                   Andhra Pr

In [31]:
# STANDARDISING TEXT COLUMNS
for df in [df_dist, df_port, df_epos]:
    df["country"] = normalize_text_column(df["country"])
    df["state"] = normalize_text_column(df["state"])

In [32]:
# SPECIFIC TEXT NORMALISATION 
for df in [df_dist, df_port, df_epos]:
    df["state"] = normalize_indian_state_names(df["state"])

In [33]:
# DATE-TIME DATA EXTRACTION (FEATURE ENGINEERING)
for df in [df_dist, df_port, df_epos]:
    df["current_year"] = extract_calendar_year(df["year_raw"])
    df["month"] = extract_month_name(df["month_raw"])
    df["financial_year"] = calculate_financial_year_from_month_raw(df["month_raw"])
    df["date"] = create_month_start_date(df["month_raw"])
    
    df["month_num"] = extract_month_number(df["date"])
    df["quarter"] = extract_quarter(df["date"])

In [34]:
# SETTING COUNTRY CODE
for df in [df_dist, df_port, df_epos]:
    df["country_code"] = map_country_to_code(df["country"])

In [35]:
# CHECKING COUNTRY
for df in [df_dist, df_port, df_epos]:
    print(df["country_code"].value_counts())

country_code
IN    2357
Name: count, dtype: int64
country_code
IN    1776
Name: count, dtype: int64
country_code
IN    2017
Name: count, dtype: int64


In [36]:
# CHECKING IF ANY STATE IS NULL?
for df in [df_dist, df_port, df_epos]:
    print(df[df["state"].isna()]["state"].unique())
    print("-"*100)
    print("\n")

[]
----------------------------------------------------------------------------------------------------


[]
----------------------------------------------------------------------------------------------------


[]
----------------------------------------------------------------------------------------------------




In [37]:
# RENAMING STATE TO STATENAME BEFORE MAPPING
df_dist = df_dist.rename(columns={"state": "state_name"})
df_port = df_port.rename(columns={"state": "state_name"})
df_epos = df_epos.rename(columns={"state": "state_name"})

In [38]:
df_dist.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'foodgrains_allocated', 'rationcards_issued',
       'trans_for_rationcards', 'aadhaar_auth_trans', 'aadhaar_trans_percent', 'epos_distributed_qty',
       'manual_distributed_qty', 'distributed_qty', 'current_year', 'month', 'financial_year', 'date', 'month_num',
       'quarter', 'country_code'],
      dtype='object')

In [39]:
df_epos.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'rationcard_holders', 'rationcards_in_trans', 'epos_trans',
       'biometric_trans', 'iris_trans', 'otp_trans', 'total_aadhaar_trans', 'total_aadhaar_trans_percent',
       'otp_auth_trans', 'not_otp_auth_trans', 'other_mode_auth_trans', 'other_mode_auth_trans_percent',
       'non_auth_trans', 'non_auth_trans_percent', 'current_year', 'month', 'financial_year', 'date', 'month_num',
       'quarter', 'country_code'],
      dtype='object')

In [40]:
df_port.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'rationcards_issued', 'interdistrict_port_aay_trans',
       'interdistrict_port_pyh_trans', 'intradistrict_port_aay_trans', 'intradistrict_port_pyh_trans',
       'interdistrict_port_aay_qty', 'interdistrict_port_pyh_qty', 'intradistrict_port_aay_qty',
       'intradistrict_port_pyh_qty', 'current_year', 'month', 'financial_year', 'date', 'month_num', 'quarter',
       'country_code'],
      dtype='object')

In [41]:
# MAPPING STATE CODES
df_dist = map_state_codes(df_dist, state_col="state_name")
df_port = map_state_codes(df_port, state_col="state_name")
df_epos = map_state_codes(df_epos, state_col="state_name")

In [42]:
df_dist.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'foodgrains_allocated', 'rationcards_issued',
       'trans_for_rationcards', 'aadhaar_auth_trans', 'aadhaar_trans_percent', 'epos_distributed_qty',
       'manual_distributed_qty', 'distributed_qty', 'current_year', 'month', 'financial_year', 'date', 'month_num',
       'quarter', 'country_code', 'state_code', 'state_num_code'],
      dtype='object')

In [43]:
df_port.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'rationcards_issued', 'interdistrict_port_aay_trans',
       'interdistrict_port_pyh_trans', 'intradistrict_port_aay_trans', 'intradistrict_port_pyh_trans',
       'interdistrict_port_aay_qty', 'interdistrict_port_pyh_qty', 'intradistrict_port_aay_qty',
       'intradistrict_port_pyh_qty', 'current_year', 'month', 'financial_year', 'date', 'month_num', 'quarter',
       'country_code', 'state_code', 'state_num_code'],
      dtype='object')

In [44]:
df_epos.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'rationcard_holders', 'rationcards_in_trans', 'epos_trans',
       'biometric_trans', 'iris_trans', 'otp_trans', 'total_aadhaar_trans', 'total_aadhaar_trans_percent',
       'otp_auth_trans', 'not_otp_auth_trans', 'other_mode_auth_trans', 'other_mode_auth_trans_percent',
       'non_auth_trans', 'non_auth_trans_percent', 'current_year', 'month', 'financial_year', 'date', 'month_num',
       'quarter', 'country_code', 'state_code', 'state_num_code'],
      dtype='object')

In [45]:
# CHECKING IF ANY STATE CODE IS NULL?
for df in [df_dist, df_port, df_epos]:
    print(df[df["state_code"].isna()]["state_name"].unique())
    print("-"*100)
    print("\n")

[]
----------------------------------------------------------------------------------------------------


[]
----------------------------------------------------------------------------------------------------


[]
----------------------------------------------------------------------------------------------------




In [46]:
for df in [df_dist, df_port, df_epos]:
    df = df.drop(columns=[
    "country",
    "year_raw",
    "month_raw",
    "state_num_code"
    ],inplace=True)

In [47]:
for df in [df_dist, df_epos, df_port]:
    print("Missing state_code:", df["state_code"].isna().sum())
    print("-"*100)
    print("\n")

Missing state_code: 0
----------------------------------------------------------------------------------------------------


Missing state_code: 0
----------------------------------------------------------------------------------------------------


Missing state_code: 0
----------------------------------------------------------------------------------------------------




In [48]:
print(df_dist.columns)

Index(['state_name', 'foodgrains_allocated', 'rationcards_issued', 'trans_for_rationcards', 'aadhaar_auth_trans',
       'aadhaar_trans_percent', 'epos_distributed_qty', 'manual_distributed_qty', 'distributed_qty', 'current_year',
       'month', 'financial_year', 'date', 'month_num', 'quarter', 'country_code', 'state_code'],
      dtype='object')


In [49]:
# REORDERING COLUMNS
desired_column_order = [
    "country_code",
    "state_name",
    "state_code",
    "current_year",
    "financial_year",
    "month",
    "month_num",
    "quarter",
    "date",
    "foodgrains_allocated",
    "rationcards_issued",
    "trans_for_rationcards",
    "aadhaar_auth_trans",
    "aadhaar_trans_percent",
    "epos_distributed_qty",
    "manual_distributed_qty",
    "distributed_qty"
]


In [50]:
existing_cols = [c for c in desired_column_order if c in df_dist.columns]
df_dist = df_dist[existing_cols]

In [51]:
print(df_epos.columns)

Index(['state_name', 'rationcard_holders', 'rationcards_in_trans', 'epos_trans', 'biometric_trans', 'iris_trans',
       'otp_trans', 'total_aadhaar_trans', 'total_aadhaar_trans_percent', 'otp_auth_trans', 'not_otp_auth_trans',
       'other_mode_auth_trans', 'other_mode_auth_trans_percent', 'non_auth_trans', 'non_auth_trans_percent',
       'current_year', 'month', 'financial_year', 'date', 'month_num', 'quarter', 'country_code', 'state_code'],
      dtype='object')


In [52]:
# REORDERING COLUMNS
desired_column_order = [
    "country_code",
    "state_name",
    "state_code",
    "current_year",
    "financial_year",
    "month",
    "month_num",
    "quarter",
    "date",
    "rationcard_holders",
    "rationcards_in_trans",
    "epos_trans",
    "biometric_trans",
    "iris_trans",
    "otp_trans",
    "total_aadhaar_trans",
    "total_aadhaar_trans_percent"
    "otp_auth_trans",
    "not_otp_auth_trans",
    "other_mode_auth_trans",
    "other_mode_auth_trans_percent",
    "non_auth_trans",
    "non_auth_trans_percent"
]

In [53]:
existing_cols = [c for c in desired_column_order if c in df_epos.columns]
df_epos = df_epos[existing_cols]

In [54]:
print(df_port.columns)

Index(['state_name', 'rationcards_issued', 'interdistrict_port_aay_trans', 'interdistrict_port_pyh_trans',
       'intradistrict_port_aay_trans', 'intradistrict_port_pyh_trans', 'interdistrict_port_aay_qty',
       'interdistrict_port_pyh_qty', 'intradistrict_port_aay_qty', 'intradistrict_port_pyh_qty', 'current_year',
       'month', 'financial_year', 'date', 'month_num', 'quarter', 'country_code', 'state_code'],
      dtype='object')


In [55]:
# REORDERING COLUMNS
desired_column_order = [
    "country_code",
    "state_name",
    "state_code",
    "current_year",
    "financial_year",
    "month",
    "month_num",
    "quarter",
    "date",
    "rationcards_issued",
    "interdistrict_port_aay_trans",
    "interdistrict_port_pyh_trans",
    "intradistrict_port_aay_trans",
    "intradistrict_port_pyh_trans",
    "interdistrict_port_aay_qty",
    "interdistrict_port_pyh_qty",
    "intradistrict_port_aay_qty",
    "intradistrict_port_pyh_qty"
]

In [56]:
existing_cols = [c for c in desired_column_order if c in df_port.columns]
df_port = df_port[existing_cols]

In [57]:
df_port.columns

Index(['country_code', 'state_name', 'state_code', 'current_year', 'financial_year', 'month', 'month_num', 'quarter',
       'date', 'rationcards_issued', 'interdistrict_port_aay_trans', 'interdistrict_port_pyh_trans',
       'intradistrict_port_aay_trans', 'intradistrict_port_pyh_trans', 'interdistrict_port_aay_qty',
       'interdistrict_port_pyh_qty', 'intradistrict_port_aay_qty', 'intradistrict_port_pyh_qty'],
      dtype='object')

In [58]:
df_dist.head(5)

,country_code,state_name,state_code,current_year,financial_year,month,month_num,quarter,date,foodgrains_allocated,rationcards_issued,trans_for_rationcards,aadhaar_auth_trans,aadhaar_trans_percent,epos_distributed_qty,manual_distributed_qty,distributed_qty
0,IN,andhra pradesh,AP,2024,2023,march,3,2024Q1,2024-03-01,154148.03,9021931.0,2809532.0,2809532.0,100.00,644.36,0.0,644.36
1,IN,arunachal pradesh,AR,2024,2023,march,3,2024Q1,2024-03-01,4677.46,180490.0,13462.0,203.0,1.51,374.46,0.0,374.46
2,IN,assam,AS,2024,2023,march,3,2024Q1,2024-03-01,135479.33,6637940.0,5029526.0,5008997.0,99.59,101321.79,0.0,101321.79
3,IN,delhi,DL,2024,2023,march,3,2024Q1,2024-03-01,37572.75,1781363.0,1161488.0,1161488.0,100.00,25696.23,0.0,25696.23
4,IN,goa,GA,2024,2023,march,3,2024Q1,2024-03-01,2856.63,128819.0,11736.0,11567.0,98.56,0.00,0.0,0.00


In [59]:
df_epos.head()

,country_code,state_name,state_code,current_year,financial_year,month,month_num,quarter,date,rationcard_holders,rationcards_in_trans,epos_trans,biometric_trans,iris_trans,otp_trans,total_aadhaar_trans,not_otp_auth_trans,other_mode_auth_trans,other_mode_auth_trans_percent,non_auth_trans,non_auth_trans_percent
0,IN,andaman and nicobar islands,AN,2024,2023,february,2,2024Q1,2024-02-01,17221.0,2869.0,2856.0,2856.0,0.0,0.0,2856.0,0.0,0.0,0.0,0.0,0.0
1,IN,andhra pradesh,AP,2024,2023,february,2,2024Q1,2024-02-01,9023477.0,8398595.0,8427589.0,8388211.0,39378.0,0.0,8427589.0,0.0,0.0,0.0,0.0,0.0
2,IN,arunachal pradesh,AR,2024,2023,february,2,2024Q1,2024-02-01,180490.0,161301.0,161332.0,11341.0,0.0,0.0,11341.0,0.0,0.0,0.0,149991.0,92.0
3,IN,assam,AS,2024,2023,february,2,2024Q1,2024-02-01,6638206.0,6242112.0,6243369.0,6197691.0,457.0,0.0,6198148.0,0.0,0.0,0.0,45221.0,0.0
4,IN,bihar,BR,2024,2023,february,2,2024Q1,2024-02-01,19327393.0,11479549.0,11536499.0,11205947.0,323201.0,0.0,11529148.0,0.0,0.0,0.0,7351.0,0.0


In [60]:
df_port.head()

,country_code,state_name,state_code,current_year,financial_year,month,month_num,quarter,date,rationcards_issued,interdistrict_port_aay_trans,interdistrict_port_pyh_trans,intradistrict_port_aay_trans,intradistrict_port_pyh_trans,interdistrict_port_aay_qty,interdistrict_port_pyh_qty,intradistrict_port_aay_qty,intradistrict_port_pyh_qty
0,IN,andaman and nicobar islands,AN,2022,2022,september,9,2022Q3,2022-09-01,16765.0,7.0,55.0,66.0,546.0,0.25,0.88,2.31,9.40
1,IN,andhra pradesh,AP,2022,2022,september,9,2022Q3,2022-09-01,8923935.0,9246.0,111199.0,159329.0,1452895.0,3037.43,11523.29,2781.91,12605.76
2,IN,arunachal pradesh,AR,2022,2022,september,9,2022Q3,2022-09-01,180490.0,0.0,0.0,0.0,0.0,0.00,0.00,0.00,0.00
3,IN,assam,AS,2022,2022,september,9,2022Q3,2022-09-01,5794113.0,17.0,138.0,5435.0,40204.0,0.60,2.56,189.79,826.15
4,IN,bihar,BR,2022,2022,september,9,2022Q3,2022-09-01,18514778.0,5667.0,42805.0,182479.0,1288410.0,195.81,1055.70,6349.19,30763.43


In [61]:
# SAVING THESE TO CSV FILES
df_dist.to_csv("../data/cleaned/distribution_data_clean.csv", index=False)
df_port.to_csv("../data/cleaned/portability_data_clean.csv", index=False)
df_epos.to_csv("../data/cleaned/epos_data_clean.csv", index=False)